# 03. pandas EDA와 누수 없는 데이터 분할

모델링 전에 데이터 계약, 결측, 중복, 그룹 구조, 레이블 분포를 확인합니다.
이 프로젝트의 두 문장은 같은 고객 사건(`case_id`)의 표현 변형일 수 있으므로 행 단위 무작위 분할은 누수를 만듭니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=sys.path.insert -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>sys.path.insert(index, path)</code></summary>

#### `sys.path.insert(index, path)`

- **역할:** Python 모듈 검색 경로의 지정 위치에 경로를 추가합니다.
- **입력·반환:** index와 경로 문자열을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 노트북에서 프로젝트 `src` 모듈을 import할 수 있게 합니다.
- **주의점:** 전역 import 상태를 바꾸므로 중복·우선순위와 실행 위치에 주의합니다.
- **공식 문서:** [sys.path.insert](https://docs.python.org/3/library/sys.html#sys.path)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    """현재 폴더부터 위로 올라가며 프로젝트 루트를 찾습니다."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "pyproject.toml과 src/가 있는 프로젝트 폴더를 찾지 못했습니다."
    )


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("project:", PROJECT_ROOT)
print("python :", sys.executable)

In [ ]:
import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

from llm_engineering_lab.data import (
    TicketSchema,
    dataset_profile,
    load_ticket_data,
    stratified_group_split,
    validate_ticket_frame,
)

DATA_PATH = PROJECT_ROOT / "data" / "customer_support_tickets.csv"
tickets = load_ticket_data(DATA_PATH)
schema = TicketSchema()
print("shape:", tickets.shape)
tickets.head()

## 1. 스키마와 dtype

ID는 숫자 계산 대상이 아니므로 문자열로, 시간은 timezone을 포함한 datetime으로 유지합니다.
`info()`는 결측과 메모리 사용량을 함께 보는 빠른 출발점입니다.

In [ ]:
tickets.info()

assert set(schema.required_columns).issubset(tickets.columns)
assert pd.api.types.is_datetime64_any_dtype(tickets[schema.created_at])

## 2. 재사용 가능한 프로파일

노트북의 일회성 출력만 보지 않고, 운영 파이프라인에서도 호출할 수 있는 `dataset_profile`을 사용합니다.

In [ ]:
profile = dataset_profile(tickets)
profile.to_dict()

## 3. 결측값: 비율과 의미를 함께 보기

실제 파일은 엄격한 검증을 통과했으므로 결측이 없습니다. 작은 복사본에 결측을 주입해 탐지와 처리 차이를 연습합니다.
결측을 무조건 제거하기 전에 “수집 실패인가, 해당 없음인가”를 구분해야 합니다.

## 4. 중복: 행 중복과 의미 중복은 다르다

`ticket_id` 중복은 데이터 계약 오류입니다. 반면 동일 `case_id`의 여러 표현은 의도된 그룹일 수 있습니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=pd.DataFrame,DataFrame.isna,Tensor.sum -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>pd.DataFrame(data=None, index=None, columns=None, ...)</code></summary>

#### `pd.DataFrame(data=None, index=None, columns=None, ...)`

- **역할:** 행·열 구조의 DataFrame을 만듭니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [pd.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

</details>

<details>
<summary><code>frame.isna()</code></summary>

#### `frame.isna()`

- **역할:** 결측값 위치를 불리언 표로 표시합니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [DataFrame.isna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
dirty = tickets.head(8).copy()
dirty.loc[1, "channel"] = pd.NA
dirty.loc[3, "text"] = ""

missing_report = (
    pd.DataFrame(
        {
            "na_count": dirty.isna().sum(),
            "blank_count": dirty.select_dtypes(include=["string", "object"]).apply(
                lambda column: column.astype("string").str.strip().eq("").sum()
            ),
        }
    )
    .fillna(0)
    .astype(int)
)
missing_report[missing_report.sum(axis=1) > 0]

In [ ]:
# 범주형 결측은 unknown으로 표시할 수 있지만, 핵심 텍스트 결측은 보통 학습에서 제외합니다.
cleaned_demo = dirty.copy()
cleaned_demo["channel"] = cleaned_demo["channel"].fillna("unknown")
cleaned_demo = cleaned_demo[cleaned_demo["text"].fillna("").str.strip().ne("")]

assert cleaned_demo["channel"].isna().sum() == 0
assert cleaned_demo["text"].str.strip().ne("").all()
cleaned_demo[["ticket_id", "channel", "text"]]

In [ ]:
duplicated_demo = pd.concat([tickets.head(3), tickets.iloc[[0]]], ignore_index=True)
duplicate_ids = duplicated_demo.loc[
    duplicated_demo["ticket_id"].duplicated(keep=False), "ticket_id"
]

assert duplicate_ids.nunique() == 1
duplicated_demo[duplicated_demo["ticket_id"].duplicated(keep=False)]

<!-- api-explanations:start -->
<!-- api-explanations:keys=DataFrame.drop_duplicates,DataFrame.groupby,Series.value_counts -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>frame.drop_duplicates(subset=None, ...)</code></summary>

#### `frame.drop_duplicates(subset=None, ...)`

- **역할:** 중복 행을 제거합니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [DataFrame.drop_duplicates](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)

</details>

<details>
<summary><code>frame.groupby(by, ...)</code></summary>

#### `frame.groupby(by, ...)`

- **역할:** key가 같은 행을 그룹으로 묶습니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [DataFrame.groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)

</details>

<details>
<summary><code>series.value_counts(normalize=False, ...)</code></summary>

#### `series.value_counts(normalize=False, ...)`

- **역할:** 고유값별 빈도를 계산합니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [Series.value_counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
deduplicated = duplicated_demo.drop_duplicates(subset="ticket_id", keep="first")
assert len(deduplicated) == len(duplicated_demo) - 1

case_sizes = tickets.groupby("case_id", observed=True).size()
print("case당 행 수 분포:")
print(case_sizes.value_counts().sort_index())

## 5. GroupBy와 교차표

평균만 보지 말고 표본 수, 범주 분포, 그룹별 편향을 함께 확인합니다.

## 6. 벡터화된 feature engineering

원본 컬럼을 덮어쓰지 않고 파생 변수를 새 DataFrame에 추가합니다.
텍스트 길이는 신호가 될 수 있지만 레이블 생성 이후에만 알 수 있는 정보는 feature로 쓰면 안 됩니다.

In [ ]:
label_summary = (
    tickets.assign(text_chars=tickets["text"].str.len())
    .groupby("label", observed=True)
    .agg(
        rows=("ticket_id", "size"),
        cases=("case_id", "nunique"),
        mean_text_chars=("text_chars", "mean"),
        high_priority_rate=("priority", lambda values: values.eq("high").mean()),
    )
    .sort_values("rows", ascending=False)
)
label_summary

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.allclose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.allclose(a, b, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `np.allclose(a, b, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 배열의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 배열 `a`, 기준 배열 `b`, 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 부동소수점 계산 결과를 정확한 값이 아니라 수치적으로 같은지 확인합니다.
- **주의점:** 조건은 `abs(a-b) <= atol + rtol*abs(b)`입니다. `==`와 달리 오차를 허용하고, `b`가 기준이라 비대칭일 수 있으며 1보다 매우 작은 값에는 기본 `atol`이 너무 클 수 있습니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [np.allclose](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
channel_by_label = pd.crosstab(
    tickets["label"],
    tickets["channel"],
    normalize="index",
).round(3)

assert np.allclose(channel_by_label.sum(axis=1), 1.0)
channel_by_label

In [ ]:
enriched = tickets.assign(
    text_chars=tickets["text"].str.len(),
    word_count=tickets["text"].str.split().str.len(),
    hour_utc=tickets["created_at"].dt.hour,
    day_of_week=tickets["created_at"].dt.day_name(),
    is_high_priority=tickets["priority"].eq("high").astype("int8"),
)

assert enriched["text_chars"].gt(0).all()
enriched[["text", "text_chars", "word_count", "hour_utc"]].head()

In [ ]:
daily_counts = (
    tickets.set_index("created_at")
    .resample("7D")["ticket_id"]
    .count()
    .rename("tickets")
)
daily_counts.head()

## 7. 시각화: 먼저 질문을 정한다

“레이블 불균형이 있는가?”를 확인하기 위한 막대그래프입니다.
`matplotlib`이 없다면 같은 집계표를 출력해 노트북 전체 흐름은 유지합니다.

In [ ]:
label_counts = tickets["label"].value_counts().sort_values()

if plt is None:
    print("matplotlib 미설치 — 집계표로 대체합니다.")
    print(label_counts)
else:
    ax = label_counts.plot.barh(figsize=(8, 4), color="#4C78A8")
    ax.set(title="Tickets per label", xlabel="rows", ylabel="label")
    ax.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    plt.show()

## 8. 데이터 계약을 실패시켜 보기

검증 함수는 필수 열, 빈 값, ID 중복, 한 case의 복수 label, timestamp를 조기에 거부합니다.
학습 중간의 모호한 오류보다 입력 경계에서 명확히 실패하는 편이 안전합니다.

In [ ]:
invalid = tickets.head(10).copy()
invalid.loc[1, "ticket_id"] = invalid.loc[0, "ticket_id"]

try:
    validate_ticket_frame(invalid, expected_labels=None)
except ValueError as error:
    assert "duplicate ticket_id" in str(error)
    print("예상한 계약 오류:", error)
else:
    raise AssertionError("중복 ID가 거부되어야 합니다.")

## 9. 행 단위 무작위 분할이 만드는 group leakage

같은 `case_id`의 두 표현이 train/validation에 갈라지면 모델이 사건 고유 표현을 기억해 성능을 과대평가할 수 있습니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=train_test_split -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>train_test_split(*arrays, test_size=None, random_state=None, ...)</code></summary>

#### `train_test_split(*arrays, test_size=None, random_state=None, ...)`

- **역할:** 배열들을 같은 index 기준으로 train과 test로 나눕니다.
- **입력·반환:** 하나 이상의 배열을 받고 각 배열의 train·test 조각을 순서대로 반환합니다.
- **이 셀에서 쓰는 이유:** 평가 데이터 누수를 막고 동일 분할을 재현합니다.
- **주의점:** 분류에서는 `stratify`와 `random_state` 필요 여부를 확인해야 합니다.
- **공식 문서:** [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from sklearn.model_selection import train_test_split

row_train, row_validation = train_test_split(
    tickets,
    test_size=0.2,
    random_state=42,
    stratify=tickets["label"],
)
leaked_cases = set(row_train["case_id"]) & set(row_validation["case_id"])

print("행 분할 train/validation 중복 case 수:", len(leaked_cases))
print("예시:", sorted(leaked_cases)[:5])

## 10. 레이블별 group 분할

프로젝트 함수는 각 레이블 안에서 `case_id`를 셔플하고 case 전체를 한쪽에 배치합니다.
행 수가 아니라 그룹 수를 기준으로 분리한다는 점이 핵심입니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Module.train -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
split = stratified_group_split(tickets, validation_size=0.2, seed=42)
train_cases = set(split.train["case_id"])
validation_cases = set(split.validation["case_id"])

split.assert_no_group_leakage()
assert train_cases.isdisjoint(validation_cases)
assert set(split.train["label"]) == set(split.validation["label"])

print("train     :", split.train.shape, "cases=", len(train_cases))
print("validation:", split.validation.shape, "cases=", len(validation_cases))

In [ ]:
split_distribution = (
    pd.concat(
        {
            "train": split.train["label"].value_counts(normalize=True),
            "validation": split.validation["label"].value_counts(normalize=True),
        },
        axis=1,
    )
    .fillna(0)
    .sort_index()
)

split_distribution["absolute_gap"] = (
    split_distribution["train"] - split_distribution["validation"]
).abs()
split_distribution

In [ ]:
if plt is None:
    print(split_distribution[["train", "validation"]])
else:
    ax = split_distribution[["train", "validation"]].plot.bar(figsize=(9, 4))
    ax.set(
        title="Label proportion after group split", ylabel="proportion", xlabel="label"
    )
    ax.legend(title="split")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()

## TODO: 재사용 가능한 품질 보고서

구현을 지운 뒤 요구사항만 보고 다시 작성하세요. 반환값은 테스트·로그에 넣기 쉬운 dict여야 합니다.

In [ ]:
def quality_report(frame: pd.DataFrame) -> dict[str, int | float]:
    # TODO 1: 행 수, case 수, 결측 셀 수, ticket_id 중복 수, 평균 텍스트 길이를 계산하세요.
    return {
        "rows": len(frame),
        "cases": int(frame["case_id"].nunique()),
        "missing_cells": int(frame.isna().sum().sum()),
        "duplicate_ticket_ids": int(frame["ticket_id"].duplicated().sum()),
        "mean_text_chars": float(frame["text"].str.len().mean()),
    }


report = quality_report(tickets)
assert report["rows"] == len(tickets)
assert report["cases"] == tickets["case_id"].nunique()
assert report["duplicate_ticket_ids"] == 0
report

## 추가 연습

- TODO 2: `channel × priority` 조합별 건수와 평균 텍스트 길이를 계산하세요.
- TODO 3: 매 레이블의 validation case가 최소 1개인지 assert를 작성하세요.
- TODO 4: `created_at` 이후에 발생한 사건만 사용하는 시간 기반 분할을 설계하고 group 누수를 검사하세요.
- TODO 5: 데이터 드리프트를 감지하기 위해 월별 channel 분포를 비교하세요.

다음 노트북에서는 이 group split을 고정한 채 scikit-learn 기준 모델을 학습합니다.